# Treinamento de classificador com BART

Notebook para treinar um modelo **BART** no arquivo `dataset.csv`.

**Esperado no CSV:**
- `Objeto` = texto
- `Relevante` = rótulo (`0` ou `1`)

Este notebook:
1. Carrega e limpa o dataset
2. Divide treino e teste
3. Tokeniza com `facebook/bart-base`
4. Treina um classificador binário
5. Avalia o modelo
6. Faz predição em novos textos


In [ ]:
# Se necessário, instale as dependências
# !pip install -q pandas numpy scikit-learn datasets transformers accelerate torch

In [1]:
import re
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import (
    BartTokenizer,
    BartForSequenceClassification,
    Trainer,
    TrainingArguments,
)

## 1. Configurações

In [4]:
DATASET_PATH = 'data/dataset/dataset.csv'   # ajuste se necessário
MODEL_NAME = 'facebook/bart-base'
OUTPUT_DIR = './bart_relevancia_model'
MAX_LENGTH = 256
TEST_SIZE = 0.2
SEED = 42

## 2. Carregar e limpar o dataset

In [5]:
df = pd.read_csv(DATASET_PATH)
df.head()

,Objeto,Relevante
0,"CAIXA, arquivo para documentos, resinada em pa...",0
1,"SACO, plastico para lixo, classe I, capacidade...",0
2,"Aquisição de medicamentos (DICLOFENACO, VITAMI...",0
3,"Aquisição de medicamentos (AMIODARONA, GLICERI...",0
4,CONTRATAÇÃO SERVIÇO DE CONFECÇÃO DE CRACHÁS PE...,1


In [6]:
# Renomear colunas esperadas
df = df.rename(columns={'Objeto': 'text', 'Relevante': 'label'})

# Garantir texto válido
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != ''].copy()

def clean_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    s = re.sub(r'\s+', '', s)

    if s == '0':
        return 0
    if s == '1':
        return 1

    # casos como '00', '0\n0', etc.
    if len(s) > 0 and set(s) == {'0'}:
        return 0
    if len(s) > 0 and set(s) == {'1'}:
        return 1

    return None

df['label'] = df['label'].apply(clean_label)
df = df[df['label'].isin([0, 1])].copy()
df['label'] = df['label'].astype(int)

print('Shape final:', df.shape)
print('\nDistribuição dos rótulos:')
print(df['label'].value_counts())

Shape final: (610, 2)

Distribuição dos rótulos:
label
0    306
1    304
Name: count, dtype: int64


In [7]:
df.sample(min(5, len(df)), random_state=SEED)

,text,label
81,"INSULINA humana regular, solução injetável 100...",0
218,Prestação de serviços de móveis e eletrodomést...,0
55,"DRONE COMPACTO DE USO PROFISSIONAL, COM CÂMERA...",0
598,Locação de Smart TV LED 65 polegadas com contr...,0
264,Aquisição de equipamentos REP-C com impressora...,1


## 3. Converter para Hugging Face Dataset

In [8]:
dataset = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
dataset = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)

train_dataset = dataset['train']
test_dataset = dataset['test']

print(train_dataset)
print(test_dataset)
print('\nExemplo de treino:')
print(train_dataset[0])

Dataset({
    features: ['text', 'label'],
    num_rows: 488
})
Dataset({
    features: ['text', 'label'],
    num_rows: 122
})

Exemplo de treino:
{'text': 'Compra de equipamentos médicos de diagnóstico com entrega imediata.', 'label': 0}


## 4. Tokenização

In [9]:
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(['text'])
test_dataset = test_dataset.remove_columns(['text'])

train_dataset.set_format('torch')
test_dataset.set_format('torch')

train_dataset

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/488 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 488
})

## 5. Modelo e métricas

In [13]:
model = BartForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # BART pode retornar uma tupla; extrair apenas os logits principais
    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

BartForSequenceClassification LOAD REPORT from: facebook/bart-base
Key                                 | Status  | 
------------------------------------+---------+-
classification_head.out_proj.bias   | MISSING | 
classification_head.out_proj.weight | MISSING | 
classification_head.dense.weight    | MISSING | 
classification_head.dense.bias      | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Argumentos de treino

In [17]:
from transformers.trainer_callback import ProgressCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Garante que apenas o ProgressCallback padrão está ativo
# (evita conflitos com callbacks de notebook em versões antigas)
for cb in trainer.callback_handler.callbacks[:]:
    if type(cb).__name__ == 'NotebookProgressCallback':
        trainer.remove_callback(cb)

if not any(isinstance(cb, ProgressCallback) for cb in trainer.callback_handler.callbacks):
    trainer.add_callback(ProgressCallback)


## 7. Treino

In [18]:
trainer.train()

  0%|          | 0/488 [00:00<?, ?it/s]

/Users/jose-cleiton/dev/script_pncp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'loss': '0.1094', 'grad_norm': '0.008619', 'learning_rate': '1.504e-05', 'epoch': '1'}


  0%|          | 0/31 [00:00<?, ?it/s]

{'eval_loss': '0.3745', 'eval_accuracy': '0.9426', 'eval_precision': '0.9661', 'eval_recall': '0.9194', 'eval_f1': '0.9421', 'eval_runtime': '2.857', 'eval_samples_per_second': '42.7', 'eval_steps_per_second': '10.85', 'epoch': '1'}


...

{'eval_loss': '0.1403', 'eval_accuracy': '0.9672', 'eval_precision': '0.9833', 'eval_recall': '0.9516', 'eval_f1': '0.9672', 'eval_runtime': '2.772', 'eval_samples_per_second': '44.01', 'eval_steps_per_second': '11.18', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '172.8', 'train_samples_per_second': '11.29', 'train_steps_per_second': '2.824', 'train_loss': '0.05064', 'epoch': '4'}


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight'].


TrainOutput(global_step=488, training_loss=0.05064375326037407, metrics={'train_runtime': 172.8161, 'train_samples_per_second': 11.295, 'train_steps_per_second': 2.824, 'total_flos': 299326758420480.0, 'train_loss': 0.05064375326037407, 'epoch': 4.0})

## 8. Avaliação

In [19]:
metrics = trainer.evaluate()
metrics

/Users/jose-cleiton/dev/script_pncp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/31 [00:00<?, ?it/s]

{'eval_loss': 0.07672438025474548,
 'eval_accuracy': 0.9672131147540983,
 'eval_precision': 0.9833333333333333,
 'eval_recall': 0.9516129032258065,
 'eval_f1': 0.9672131147540983,
 'eval_runtime': 3.4203,
 'eval_samples_per_second': 35.669,
 'eval_steps_per_second': 9.064,
 'epoch': 4.0}

In [20]:
pred_output = trainer.predict(test_dataset)
y_true = pred_output.label_ids

# BART pode retornar uma tupla em predictions; extrair apenas os logits principais
predictions = pred_output.predictions
if isinstance(predictions, (tuple, list)):
    predictions = predictions[0]

y_pred = np.argmax(predictions, axis=-1)

print('Classification report:')
print(classification_report(y_true, y_pred, digits=4, zero_division=0))

print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))


/Users/jose-cleiton/dev/script_pncp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/31 [00:00<?, ?it/s]

Classification report:
              precision    recall  f1-score   support

           0     0.9516    0.9833    0.9672        60
           1     0.9833    0.9516    0.9672        62

    accuracy                         0.9672       122
   macro avg     0.9675    0.9675    0.9672       122
weighted avg     0.9677    0.9672    0.9672       122

Confusion matrix:
[[59  1]
 [ 3 59]]


## 9. Salvar modelo

In [21]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Modelo salvo em: {OUTPUT_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: ./bart_relevancia_model


## 10. Inferência em novos textos

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def predict_texts(texts):
    inputs = tokenizer(
        texts,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)

    results = []
    for text, pred, prob in zip(texts, preds, probs):
        results.append({
            'texto': text,
            'relevante': int(pred),
            'prob_nao_relevante': float(prob[0]),
            'prob_relevante': float(prob[1]),
        })
    return pd.DataFrame(results)

In [23]:
novos_textos = [
    'Contratação de software para controle de acesso com reconhecimento facial',
    'Aquisição de medicamentos hospitalares',
    'Prestação de serviço de videomonitoramento com câmeras IP e NVR',
    'Compra de cadeiras escolares',
    'Relógio de ponto biométrico com impressora térmica integrada',
]

predict_texts(novos_textos)

,texto,relevante,prob_nao_relevante,prob_relevante
0,Contratação de software para controle de acess...,1,0.000048,0.999952
1,Aquisição de medicamentos hospitalares,0,0.999888,0.000112
2,Prestação de serviço de videomonitoramento com...,1,0.000007,0.999992
3,Compra de cadeiras escolares,0,0.999970,0.000030
4,Relógio de ponto biométrico com impressora tér...,1,0.000011,0.999989


## 11. Observações

- `facebook/bart-base` funciona, mas é um checkpoint base e não específico para português.
- Depois vale comparar com modelos como **BERTimbau**.
- Se quiser testar sem treino, uma opção é **`facebook/bart-large-mnli`** para zero-shot.